In [ ]:
import os
import shutil
import glob
import pandas as pd
from typing import Literal
import warnings

warnings.filterwarnings('ignore', category=RuntimeWarning)

SAMPLE_DATA_FILE = r"D:\Data\TCS\TCdata.xlsx"
foto_dir = r"D:\Data\raw"

SE_or_VG: Literal["SE", "VG", "Both"] = "Both"
DISABLE_ALL_FILTERS = False
PRE_BIAS_THRESH = 0.5

CYCLE_ORDER = ["P", "E", "M", "D"]
IMG_SUFFIX = [".png", ".tif"]


# ==========================================
# 1. LOAD & FILTER DATA
# ==========================================
df = pd.read_excel(SAMPLE_DATA_FILE, sheet_name='wt-total')

# Filter data if needed
if not DISABLE_ALL_FILTERS:
    if SE_or_VG == "SE":
        df = df[df["Virgin"] != 1]
    elif SE_or_VG == "VG":
        df = df[df["Virgin"] == 1]

    df['PB'] = (df['Di_pre'] + df['Do_pre'] - df['Si_pre'] - df['So_pre'])/(df['Di_pre'] + df['Do_pre'] + df['Si_pre'] + df['So_pre'])
    df = df[abs(df['PB']) <= PRE_BIAS_THRESH]

sorted_dir = os.path.join(foto_dir, "sorted")

if os.path.isdir(sorted_dir):
    shutil.rmtree(sorted_dir)

folder_dict = {}

for c in CYCLE_ORDER:
    folder_dict[c] = os.path.join(sorted_dir, c)
    os.makedirs(folder_dict[c], exist_ok=True)


print(f"\nStarting image sorting for {len(df)} valid subjects...")

stats = {c: 0 for c in CYCLE_ORDER}
not_found = []

for _, row in df.iterrows():
    exp_id = str(row['Exp']).strip()
    cycle_stage = str(row['Cycle']).strip()
    fem_id = str(row['Fem']).strip().replace("N", "")
    
    if cycle_stage not in CYCLE_ORDER:
        continue

    exp_date = exp_id[:4]

    src_folder = os.path.join(foto_dir, f"2026{exp_date}")
    dest_folder = folder_dict[cycle_stage]
    files_moved_for_this_exp = 0

    for suffix in IMG_SUFFIX:
        print(f"{fem_id}{suffix}")
        search_pattern = os.path.join(src_folder, f"*{fem_id}{suffix}")
        matching_files = glob.glob(search_pattern)
        
        for src_file in matching_files:
            filename = f"{exp_date}_{os.path.basename(src_file)}"
            dest_path = os.path.join(dest_folder, filename)

            shutil.copy2(src_file, dest_path)
            files_moved_for_this_exp += 1
            
    if files_moved_for_this_exp > 0:
        stats[cycle_stage] += files_moved_for_this_exp
    else:
        not_found.append(exp_id)

# ==========================================
# 3. REPORT RESULTS
# ==========================================
print("\n✅ Sorting Complete!")
print("-" * 40)
for stage in CYCLE_ORDER:
    print(f"Stage {stage}: {stats[stage]} files sorted into {folder_dict[stage]}")

if not_found:
    print(f"\n⚠️ Warning: No images found for {len(not_found)} Exp IDs:")
    print(", ".join(not_found[:10]) + ("..." if len(not_found) > 10 else ""))

